In [ ]:
!pip install qiskit qiskit-aer --quiet

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

from qiskit.quantum_info import Statevector, state_fidelity, partial_trace

from qiskit_aer.noise import NoiseModel, depolarizing_error

os.makedirs("outputs", exist_ok=True)

print("Quantum Teleportation Notebook Ready ✅")

Quantum Teleportation Notebook Ready ✅


In [ ]:
def create_teleportation_unitary(theta=np.pi/4, phi=0):
    """
    Teleportation circuit using coherent correction (no c_if).
    Works in all Qiskit versions.
    """

    qc = QuantumCircuit(3)

    # Step 1: Prepare message state on qubit 0
    qc.ry(theta, 0)
    qc.rz(phi, 0)
    qc.barrier()

    # Step 2: Create Bell pair between qubit 1 and qubit 2
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()

    # Step 3: Alice entangles message with Bell qubit
    qc.cx(0, 1)
    qc.h(0)
    qc.barrier()

    # Step 4: Bob applies coherent corrections (CX + CZ)
    qc.cx(1, 2)
    qc.cz(0, 2)
    qc.barrier()

    return qc

In [ ]:
def verify_fidelity(theta=np.pi/4, phi=0):

    # Build teleportation circuit
    qc = create_teleportation_unitary(theta, phi)

    # Initial message state alone
    msg = QuantumCircuit(1)
    msg.ry(theta, 0)
    msg.rz(phi, 0)
    init_state = Statevector.from_instruction(msg)

    # Final 3-qubit state
    final_state = Statevector.from_instruction(qc)

    # Extract Bob’s qubit reduced state
    bob_state = partial_trace(final_state, [0, 1])

    # Fidelity with original message state
    fidelity = state_fidelity(bob_state, init_state)

    return fidelity

In [ ]:
def simulate_with_noise(qc, shots=2048):

    noise_model = NoiseModel()

    # Define noise strength
    error_1q = depolarizing_error(0.01, 1)
    error_2q = depolarizing_error(0.05, 2)

    # Apply noise to gates
    noise_model.add_all_qubit_quantum_error(error_1q, ["h", "x", "z"])
    noise_model.add_all_qubit_quantum_error(error_2q, ["cx", "cz"])

    sim = AerSimulator(noise_model=noise_model)

    # Add measurement for histogram output
    qc_meas = qc.copy()
    qc_meas.measure_all()

    result = sim.run(qc_meas, shots=shots).result()
    return result.get_counts()

In [ ]:
def simulate_ideal(qc, shots=2048):

    sim = AerSimulator()

    qc_meas = qc.copy()
    qc_meas.measure_all()

    result = sim.run(qc_meas, shots=shots).result()
    return result.get_counts()

In [ ]:
theta = np.pi/4
phi = 0

qc = create_teleportation_unitary(theta, phi)

print("Teleportation Circuit:\n")
print(qc.draw())

Teleportation Circuit:

     ┌─────────┐┌───────┐ ░            ░      ┌───┐ ░          ░ 
q_0: ┤ Ry(π/4) ├┤ Rz(0) ├─░────────────░───■──┤ H ├─░───────■──░─
     └─────────┘└───────┘ ░ ┌───┐      ░ ┌─┴─┐└───┘ ░       │  ░ 
q_1: ─────────────────────░─┤ H ├──■───░─┤ X ├──────░───■───┼──░─
                          ░ └───┘┌─┴─┐ ░ └───┘      ░ ┌─┴─┐ │  ░ 
q_2: ─────────────────────░──────┤ X ├─░────────────░─┤ X ├─■──░─
                          ░      └───┘ ░            ░ └───┘    ░ 


In [ ]:
shots = 2048

counts_ideal = simulate_ideal(qc, shots)
counts_noisy = simulate_with_noise(qc, shots)

print("Ideal Counts:", counts_ideal)
print("Noisy Counts:", counts_noisy)

Ideal Counts: {'100': 73, '101': 89, '111': 79, '011': 434, '001': 431, '110': 70, '010': 443, '000': 429}
Noisy Counts: {'101': 91, '110': 112, '100': 97, '111': 105, '011': 419, '001': 379, '000': 416, '010': 429}


In [ ]:
plot_histogram(
    [counts_ideal, counts_noisy],
    legend=["Ideal", "Noisy"]
)

plt.savefig("outputs/teleportation_histogram.png", dpi=300)
plt.show()

print("Saved plot: outputs/teleportation_histogram.png")

<Figure size 640x480 with 0 Axes>

Saved plot: outputs/teleportation_histogram.png


In [ ]:
fidelity = verify_fidelity(theta, phi)

print("\nTeleportation Fidelity:", fidelity)

if fidelity > 0.99:
    print("✅ Perfect Teleportation Achieved")
else:
    print("⚠ Fidelity Reduced Due to Noise")


Teleportation Fidelity: 0.9999999999999994
✅ Perfect Teleportation Achieved
